# ESM-2 Embeddings for Marine Metallothionein Sequences

Generates mean-pooled per-sequence embeddings from ESM-2 protein language model.
Upload `sequences.fasta` from `data/processed/sequences.fasta` before running.

Output: `esm2_embeddings.csv` - download and place in `data/processed/`.

In [ ]:
# Install fair-esm
!pip install fair-esm -q

In [ ]:
import torch
import esm
import csv
import io
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Choose model based on available VRAM:
#   T4  (15 GB) -> esm2_t33_650M_UR50D  (1280-dim, fits easily)
#   A100 (40 GB) -> esm2_t36_3B_UR50D   (2560-dim, recommended for publication)
#
# Change MODEL_NAME to match your Colab GPU.

MODEL_NAME = "esm2_t36_3B_UR50D"   # change to esm2_t33_650M_UR50D for T4

print(f"Loading {MODEL_NAME} ...")
model, alphabet = esm.pretrained.load_model_and_alphabet_hub(MODEL_NAME)
model = model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

# Embedding dimension
EMB_DIM = model.embed_dim
print(f"Embedding dim: {EMB_DIM}")

In [ ]:
# Upload sequences.fasta when prompted
from google.colab import files
uploaded = files.upload()   # select sequences.fasta
fasta_text = list(uploaded.values())[0].decode("utf-8")

In [ ]:
def parse_fasta(text):
    seqs = []
    uid, buf = None, []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith(">"):
            if uid:
                seqs.append((uid, "".join(buf)))
            uid, buf = line[1:].strip(), []
        else:
            buf.append(line)
    if uid:
        seqs.append((uid, "".join(buf)))
    return seqs

sequences = parse_fasta(fasta_text)
print(f"Parsed {len(sequences)} sequences")
print(f"Length range: {min(len(s) for _,s in sequences)}-{max(len(s) for _,s in sequences)} aa")

In [ ]:
# ESM-2 has a 1022 residue limit per sequence. MTs are 40-130 aa, so no truncation needed.
# Batch size: 16 fits comfortably on T4/A100 for sequences this short.

BATCH_SIZE = 16
MAX_LEN = 1022

all_ids = []
all_embeddings = []

for batch_start in range(0, len(sequences), BATCH_SIZE):
    batch = sequences[batch_start : batch_start + BATCH_SIZE]
    # Truncate just in case (MTs won't trigger this)
    batch_data = [(uid, seq[:MAX_LEN]) for uid, seq in batch]

    _, _, tokens = batch_converter(batch_data)
    tokens = tokens.to(device)

    with torch.no_grad():
        results = model(tokens, repr_layers=[model.num_layers], return_contacts=False)

    # repr_layers returns shape [batch, seq_len+2, embed_dim] (includes BOS/EOS tokens)
    token_reps = results["representations"][model.num_layers]

    for i, (uid, seq) in enumerate(batch):
        # Mean-pool over residue positions only (exclude BOS token at 0 and EOS)
        seq_len = len(seq[:MAX_LEN])
        emb = token_reps[i, 1 : seq_len + 1].mean(dim=0).cpu().float().numpy()
        all_ids.append(uid)
        all_embeddings.append(emb)

    if (batch_start // BATCH_SIZE) % 5 == 0:
        print(f"[{batch_start + len(batch)}/{len(sequences)}] done")

print(f"Generated {len(all_embeddings)} embeddings of dim {all_embeddings[0].shape[0]}")

In [ ]:
# Save to CSV: uniprot_id, esm2_0, esm2_1, ..., esm2_{EMB_DIM-1}
import numpy as np

col_names = [f"esm2_{i}" for i in range(all_embeddings[0].shape[0])]
out_path = "esm2_embeddings.csv"

with open(out_path, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["uniprot_id"] + col_names)
    for uid, emb in zip(all_ids, all_embeddings):
        w.writerow([uid] + emb.tolist())

print(f"Saved {len(all_ids)} rows x {len(col_names)+1} cols -> {out_path}")

In [ ]:
# Download the embeddings CSV
from google.colab import files
files.download("esm2_embeddings.csv")